# Task B Run 22 -- package Run 21's Gemma-4-12B predictions (CPU, ~1 minute)

Run 21 trained everything. Both Gemma-4-12B full fits finished, but its last cell looked
for the wrong filename (`test_probs.npy` instead of `test_inputs_probs.npy`), so only the
MuRIL ZIP was written. The probabilities are in Run 21's output. This notebook reads them
back and writes the ZIPs. Nothing is retrained.

Run 21's holdout result (530 rows, fingerprint `f85f4f049b`):

| arm | macro-F1 |
|---|---|
| MuRIL, Run 9 recipe | 0.6151 |
| **Gemma-4-12B, 2 seeds** | **0.6792** |
| 50/50 ensemble | 0.6825 |

Gemma minus MuRIL is **+0.064, 95% CI [+0.024, +0.105]**. Ensemble minus Gemma is +0.003,
CI [-0.019, +0.025], so the rule fixed in advance recommends the **LLM alone**.

**To run:**
1. **Add Input -> Notebook Output -> your Run 21 notebook.**
2. Accelerator: **None** (CPU is enough). Internet: **on**, for the repo.
3. **Save & Run All.** The ZIPs are written to `/kaggle/working/run21_submissions/`.

In [ ]:
import glob, os, pathlib, subprocess, sys

REPO = "https://github.com/robinpnalex/Hastika-ICON2026.git"
WORK = "/kaggle/working/hastika"
if not os.path.isdir(WORK + "/.git"):
    subprocess.run(["git", "clone", "-q", "-b", "task-b", "--depth", "1", REPO, WORK], check=True)
subprocess.run(["git", "-C", WORK, "fetch", "-q", "--depth", "1", "origin", "task-b"], check=True)
subprocess.run(["git", "-C", WORK, "reset", "-q", "--hard", "FETCH_HEAD"], check=True)
subprocess.run("pip install -q emoji ftfy", shell=True, check=True)

found = sorted(glob.glob("/kaggle/input/**/b21_outputs/result.json", recursive=True))
assert found, "attach Run 21's output: Add Input -> Notebook Output"
B21 = pathlib.Path(found[0]).parent
print("Run 21 output:", B21)
print(sorted(p.name for p in (B21 / "runs").iterdir()))
subprocess.run([sys.executable, "-m", "hastika.task_b.package_b21", "--b21", str(B21),
                "--out", "/kaggle/working/run21_submissions"],
               check=True, cwd=WORK, env={**os.environ, "PYTHONPATH": f"{WORK}/src"})

Submit `RECOMMENDED_b21_llm.zip`. Check the printed predicted/prior class shares first: a
class that is almost never predicted forfeits most of its F1. Record the CodaBench score
in `docs/EXPERIMENTS.md` beside the holdout numbers. The current best is `0.6410`.